# Stage 1b – QLoRA fine-tuning on transcript SUMMARIES
Same model, split, training events and hyper-parameters as Stage 1. **Only change:** the transcript in every prompt is the
extractive summary from `llm_data_sum.jsonl.gz` (key sentences from prepared remarks + Q&A, ~2,200 tokens) instead of the
first ~3k tokens of the call. Scoring uses fp32 for the final Long/Short step (no fp16 ties).

**Run order (Kaggle, GPU T4 x2, Internet ON, inputs: the summary dataset + `llm_stage1.py`):**
1. `MODE = "smoke"`  → ~30 min. Trains 20 steps on 160 events, scores 50 validation events, prints all checks and the time estimate.
2. `MODE = "train_eval"` → ~8 h. Full training on 3,000 events, saves the adapter, scores validation + test in fp32.
Set `DATASET` to `"base"` or `"full"`; everything else stays as it is.


In [ ]:
%%capture
# Official Unsloth install cells (unslothai/notebooks, Llama3 8B) – pinned versions that are known to work together
import os
if "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle"):
    !pip install pip3-autoremove
    !pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
    !pip install unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install transformers==4.57.6
    !pip install --no-deps trl==0.22.2
else:  # Google Colab
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate peft trl triton unsloth
    !unsloth install-kernels
    !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install transformers==4.57.6
    !pip install --no-deps trl==0.22.2
!pip install -q scikit-learn

## 1) Configuration – the only cell you edit

In [ ]:
DATASET     = "base"      # "base" or "full"  (paper's two datasets – run once each)
MODE        = "smoke"     # "smoke"      -> 20 training steps on 160 events + scoring 50 val events: checks everything (~30 min)
                          # "timing"     -> 10 training steps, prints how long the full run will take
                          # "pilot"      -> quick check: trains on PILOT_N events, scores VALIDATION only (test untouched)
                          # "train_eval" -> fine-tune (1 epoch) + evaluate on validation & test
                          # "zeroshot"   -> NO fine-tuning: plain Llama-3-8B-Instruct baseline (paper's GPT-4 row)
SPLIT_MODE  = "time"      # "time" (main: train<=2023-06, val 2023-H2, test 2024-H1) or "random" (paper-like)
MAX_SEQ_LEN = 3072        # tokens per example (paper 25k; T4 realistic 2048-4096) - keep SAME for base & full
TRAIN_LIMIT = 3000        # decided from the timing run (all 6,387 would take ~17 h > Kaggle 12 h limit)
VAL_LIMIT   = 500         # validation events used to choose the decision threshold
SEED        = 3407
BALANCE_TRAIN = True     # train on a 50/50 Long/Short subset (pilot showed the model drifting to the majority class)
PILOT_N     = 400         # training events used in MODE="pilot"
TRAIN_BUDGET_HOURS = 8.9 # safety: stop training at 8.9 h and still evaluate (Kaggle kills runs at 12 h)

# paper hyper-parameters (Ni et al. 2024, Fine-tuning section)
EPOCHS, LR, OPTIM, WEIGHT_DECAY, SCHEDULER, WARMUP = 1, 2e-4, "adamw_8bit", 0.01, "linear", 5
BATCH, GRAD_ACC = 1, 8    # effective batch 8 (paper: not stated)
LORA_R, LORA_ALPHA = 16, 16   # paper: not stated -> Unsloth defaults
MODEL_NAME = "unsloth/llama-3-8b-Instruct-bnb-4bit"
DATA_FILE = "llm_data_sum.jsonl.gz"   # summarized transcripts (Stage 1b) - NOT the old llm_data.jsonl
SMOKE_TRAIN, SMOKE_VAL = 160, 50
RUN_NAME = f"{DATASET}_{SPLIT_MODE}_{MAX_SEQ_LEN}_{MODE}_sum"
print(RUN_NAME)

## 2) Data

In [ ]:
import os, sys, glob, json, time, random
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # Unsloth trains on one GPU
import numpy as np

# Optional fallback if the Kaggle dataset fails to mount: Google Drive file IDs ("Anyone with the link")
DATA_FILE_ID = ""      # id of llm_data.jsonl.gz   (leave "" when using the Kaggle dataset)
CODE_FILE_ID = ""      # id of llm_stage1.py

DL = "/kaggle/working/data_dl" if os.path.exists("/kaggle/working") else "data_dl"
os.makedirs(DL, exist_ok=True)
print("Files under /kaggle/input:")
for p in glob.glob("/kaggle/input/**/*", recursive=True):
    if os.path.isfile(p): print("  ", p, round(os.path.getsize(p)/1e6, 1), "MB")

def find(*names):
    for name in names:
        hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"{DL}/{name}") + glob.glob(name)
        if hits: return hits[0]
    return None

DATA_PATH = find(DATA_FILE, DATA_FILE.replace(".gz", ""))
CODE_PATH = find("llm_stage1.py")
if (DATA_PATH is None or CODE_PATH is None) and DATA_FILE_ID:
    os.system("pip install -q -U gdown")
    import gdown
    DATA_PATH = DATA_PATH or gdown.download(id=DATA_FILE_ID, output=f"{DL}/llm_data.jsonl.gz", quiet=False)
    CODE_PATH = CODE_PATH or gdown.download(id=CODE_FILE_ID, output=f"{DL}/llm_stage1.py", quiet=False)
assert DATA_PATH and CODE_PATH, "data/code not found: add the Kaggle dataset as Input (or fill the Drive IDs)"
print("data:", DATA_PATH, "\ncode:", CODE_PATH)

import importlib.util
spec = importlib.util.spec_from_file_location("llm_stage1", CODE_PATH)
L = importlib.util.module_from_spec(spec); spec.loader.exec_module(L)
OUT = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUT, exist_ok=True)

def balanced_subsample(recs, n, seed):
    """n events, half Long / half Short, deterministic (same events for Base and Full)."""
    longs  = L.subsample([r for r in recs if r["label"] == "Long"],  None, seed)
    shorts = L.subsample([r for r in recs if r["label"] == "Short"], None, seed)
    rs = sorted(longs, key=lambda r: r["event_id"]); random.Random(seed).shuffle(rs); longs = rs
    rs = sorted(shorts, key=lambda r: r["event_id"]); random.Random(seed + 1).shuffle(rs); shorts = rs
    k = min(len(longs), len(shorts)) if n is None else n // 2
    out = longs[:k] + shorts[:k]
    random.Random(seed).shuffle(out)
    return out

records = L.load_records(DATA_PATH)
assert len(records) == 8271, f"expected 8271 rows, got {len(records)}"
splits = L.split_records(records, SPLIT_MODE, seed=SEED)
N_TRAIN = {"pilot": PILOT_N, "smoke": SMOKE_TRAIN}.get(MODE, TRAIN_LIMIT)
train_recs = balanced_subsample(splits["train"], N_TRAIN, SEED) if BALANCE_TRAIN else L.subsample(splits["train"], N_TRAIN, seed=SEED)
val_recs   = L.subsample(splits["val"], VAL_LIMIT, seed=SEED)
test_recs  = splits["test"]
if MODE == "smoke":
    val_recs = val_recs[:SMOKE_VAL]
for n, r in [("train", train_recs), ("val", val_recs), ("test", test_recs)]:
    print(f"{n:5s}: {len(r):5d} events | Long share {np.mean([x['label']=='Long' for x in r]):.3f}")

# --- checks: this must be the SUMMARY file, unchanged otherwise
n_sum = sum("Prepared remarks (key sentences):" in r["transcript"] for r in records)
print(f"rows with a summary: {n_sum}/{len(records)} | summary chars median {int(np.median([len(r['transcript']) for r in records]))}")
assert n_sum == len(records), "this is not the summary file - add the llm_data_sum.jsonl.gz dataset as input"


In [ ]:
# Reporting helpers: every decision uses VALIDATION only; the test set is scored once.
from sklearn.metrics import roc_auc_score
import pandas as pd

def report(y_val, p_val, y_test=None, p_test=None, train_long_share=0.5):
    """argmax_0.5   : model's own answer (paper-comparable)
       calibrated   : threshold = validation quantile so that the share of 'Long' predictions equals the
                      training Long share (removes the model's constant bias toward one class)
       val_tuned    : threshold maximising validation MCC
       auc          : ranking quality, independent of any threshold (0.5 = random)"""
    p_val = np.asarray(p_val)
    thr_cal = float(np.quantile(p_val, 1 - train_long_share))
    thr_mcc = L.pick_threshold(y_val, p_val)
    res = {"thresholds": {"calibrated": thr_cal, "val_tuned": thr_mcc},
           "p_spread_val": [float(p_val.min()), float(np.median(p_val)), float(p_val.max())],
           "val": {"argmax_0.5": L.evaluate(y_val, p_val, 0.5), "calibrated": L.evaluate(y_val, p_val, thr_cal),
                   "auc": float(roc_auc_score([y == "Long" for y in y_val], p_val))}}
    if y_test is not None:
        res["test"] = {"argmax_0.5": L.evaluate(y_test, p_test, 0.5),
                       "calibrated": L.evaluate(y_test, p_test, thr_cal),
                       "val_tuned":  L.evaluate(y_test, p_test, thr_mcc),
                       "auc": float(roc_auc_score([y == "Long" for y in y_test], p_test))}
    return res

def show(res):
    for part in ("val", "test"):
        if part not in res: continue
        print(f"--- {part.upper()}  (AUC {res[part]['auc']:.3f})")
        for k in ("argmax_0.5", "calibrated", "val_tuned"):
            if k in res[part]:
                v = res[part][k]
                print(f"  {k:11s} acc {v['accuracy']:.3f} [{v['ci95']['accuracy'][0]:.3f}-{v['ci95']['accuracy'][1]:.3f}]  F1w {v['f1_weighted']:.3f}  MCC {v['mcc']:.3f}  pred-Long {v['pred_long_share']:.2f}")
    print("p_long (val) min/median/max:", ["%.3f" % x for x in res["p_spread_val"]])

def save_preds(name, recs, p):
    pd.DataFrame({"event_id": [r["event_id"] for r in recs], "label": [r["label"] for r in recs], "p_long": p}).to_csv(
        f"{OUT}/pred_{name}_{RUN_NAME}.csv", index=False)

TRAIN_LONG_SHARE = float(np.mean([r["label"] == "Long" for r in train_recs])) if MODE != "zeroshot" else \
                   float(np.mean([r["label"] == "Long" for r in splits["train"]]))
print("train Long share used for calibration:", round(TRAIN_LONG_SHARE, 3))


## 3) Model (4-bit) + Llama-3 chat template

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)
tokenizer = get_chat_template(tokenizer, chat_template="llama-3")
print("answer token ids:", L.label_token_ids(tokenizer))
gpu = torch.cuda.get_device_properties(0); print(gpu.name, round(gpu.total_memory/1e9, 1), "GB")

In [ ]:
# Build prompts: instruction + all textualised features (never cut) + transcript (tail cut to fit)
t0 = time.time()
val_prompts,  st_val  = L.build_texts(val_recs,  DATASET, tokenizer, MAX_SEQ_LEN, with_answer=False)
test_prompts, st_test = L.build_texts(test_recs, DATASET, tokenizer, MAX_SEQ_LEN, with_answer=False)
print("val :", st_val); print("test:", st_test)
print(f"({time.time()-t0:.0f}s)")
print(test_prompts[0][:1500], "...")
# --- check: summaries must fit without cutting
for nm, st in (("val", st_val), ("test", st_test)):
    assert st["truncated_share"] <= 0.02, f"{nm}: {st['truncated_share']:.1%} of prompts were cut - inspect the training log before continuing"
print("OK: prompts fit in", MAX_SEQ_LEN, "tokens")


In [ ]:
# ---- fp32 scoring of the final Long/Short step (fixes the fp16 ties seen in Stage 1)
# fp32 scoring: the training run computed the final Long/Short logits in fp16. Logits of size ~16-32 can
# only move in steps of 1/64 in fp16, so P(Long) could only take values 0.5 +- k/256 and ~30% of events
# came out as an exact 0.500 tie. Here the last hidden state is taken from the model and ONLY the two answer
# rows of the output layer are multiplied in float32 -> the model's true, untied P(Long).


def _parts(model):
    base = model.get_base_model() if hasattr(model, "get_base_model") else model   # PeftModel -> LlamaForCausalLM
    return base, base.model, base.get_output_embeddings()


def _answer_rows_fp32(head, token_ids):
    W = head.weight
    if hasattr(W, "quant_state"):                       # only if the output layer itself were 4-bit
        import bitsandbytes.functional as bnbF
        W = bnbF.dequantize_4bit(W.data, W.quant_state)
    return W[token_ids].detach().float()                # [2, hidden]


def _last_hidden_inner(base, inner, head, enc, apply_norm):
    """Strategy A: run the decoder body directly and take the last position's hidden state."""
    out = inner(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], use_cache=False)
    h = out[0][:, -1:, :]
    if apply_norm:
        h = inner.norm(h)
    return h[0, -1]


def _last_hidden_hook(base, inner, head, enc, apply_norm):
    """Strategy B: run the full model as in training-time scoring and capture the input of the output layer."""
    got = {}
    hook = head.register_forward_hook(lambda m, args, out: got.__setitem__("h", args[0]))
    try:
        with torch.inference_mode():
            try:
                base(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"], logits_to_keep=1)
            except TypeError:
                base(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    finally:
        hook.remove()
    if "h" not in got:
        raise RuntimeError("output layer was not called through its module")
    return got["h"][0, -1]


STRATEGIES = {"inner": (_last_hidden_inner, False), "inner+norm": (_last_hidden_inner, True),
              "hook": (_last_hidden_hook, False)}


def score_prompts_fp32(model, tokenizer, prompts, token_ids, strategy="inner", log_every=100):
    base, inner, head = _parts(model)
    W2 = _answer_rows_fp32(head, token_ids).to(base.device if hasattr(base, "device") else "cpu")
    fn, norm = STRATEGIES[strategy]
    model.eval()
    probs, t0 = [], time.time()
    for i, p in enumerate(prompts):
        enc = tokenizer(p, add_special_tokens=False, return_tensors="pt").to(W2.device)
        with torch.inference_mode():
            h = fn(base, inner, head, enc, norm)
            z = W2 @ h.float()                          # [logit_Long, logit_Short] in float32
            probs.append(torch.softmax(z, 0)[0].item())
        if log_every and (i + 1) % log_every == 0:
            el = time.time() - t0
            print(f"  scored {i + 1}/{len(prompts)}  ({el / (i + 1):.2f}s each, "
                  f"~{el / (i + 1) * (len(prompts) - i - 1) / 60:.0f} min left)", flush=True)
    return np.array(probs)


def pick_strategy(model, tokenizer, prompts, token_ids, old_scorer, tol=0.0125):
    """Check on a few prompts that the fp32 score equals the old fp16 score up to fp16 rounding
    (steps of 1/256). Returns the first strategy that agrees; stops the run if none does."""
    p_old = np.asarray(old_scorer(prompts))
    print("old fp16 p_long :", np.round(p_old, 5))
    for name in STRATEGIES:
        try:
            p_new = score_prompts_fp32(model, tokenizer, prompts, token_ids, name, log_every=0)
        except Exception as e:                          # noqa: BLE001 - try the next strategy
            print(f"strategy {name!r} failed: {type(e).__name__}: {e}")
            continue
        diff = float(np.max(np.abs(p_new - p_old)))
        print(f"strategy {name!r}: fp32 p_long {np.round(p_new, 5)}  max diff vs fp16 {diff:.5f}")
        if diff <= tol:
            print(f"-> using {name!r} (agrees with the fp16 scores, only finer)")
            return name
    raise RuntimeError("fp32 scores do not match the original fp16 scores - inspect the scoring log before continuing")

_STRATEGY = None
def SCORE(prompts):
    global _STRATEGY
    ids = L.label_token_ids(tokenizer); tid = [ids["Long"], ids["Short"]]
    if _STRATEGY is None:   # one-time check on 8 prompts: fp32 must agree with the old fp16 method, only finer
        _STRATEGY = pick_strategy(model, tokenizer, prompts[:8], tid, lambda ps: L.score_prompts(model, tokenizer, ps, log_every=0))
    return score_prompts_fp32(model, tokenizer, prompts, tid, _STRATEGY)

## 4a) Zero-shot baseline (MODE = "zeroshot") – the paper's "no fine-tuning" comparison

In [ ]:
if MODE == "zeroshot":
    FastLanguageModel.for_inference(model)
    y_val, y_test = [r["label"] for r in val_recs], [r["label"] for r in test_recs]
    p_val  = SCORE(val_prompts)
    p_test = SCORE(test_prompts)
    save_preds("val", val_recs, p_val); save_preds("test", test_recs, p_test)
    res = report(y_val, p_val, y_test, p_test, TRAIN_LONG_SHARE)
    res["run"] = RUN_NAME
    json.dump(res, open(f"{OUT}/results_{RUN_NAME}.json", "w"), indent=2)
    show(res)
    print(L.comparison_table({("Llama-3-8B-Instruct zero-shot (ours, argmax)", DATASET): res["test"]["argmax_0.5"],
                              ("Llama-3-8B-Instruct zero-shot (ours, calibrated)", DATASET): res["test"]["calibrated"]}))

## 4b) QLoRA fine-tuning (MODE = "timing" or "train_eval")

In [ ]:
if MODE in ("timing", "train_eval", "pilot", "smoke"):
    from trl import SFTConfig, SFTTrainer
    from transformers import DataCollatorForSeq2Seq
    from datasets import Dataset
    from unsloth.chat_templates import train_on_responses_only

    train_texts, st_train = L.build_texts(train_recs, DATASET, tokenizer, MAX_SEQ_LEN, with_answer=True)
    print("train:", st_train)
    train_ds = Dataset.from_dict({"text": train_texts}).shuffle(seed=SEED)

    model = FastLanguageModel.get_peft_model(
        model, r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=SEED, use_rslora=False, loftq_config=None)

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_ds,
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN, packing=False,
        data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
        args=SFTConfig(
            per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC,
            num_train_epochs=EPOCHS, max_steps=10 if MODE == "timing" else -1,
            learning_rate=LR, optim=OPTIM, weight_decay=WEIGHT_DECAY,
            lr_scheduler_type=SCHEDULER, warmup_steps=WARMUP,
            logging_steps=10, logging_first_step=True, save_strategy="steps", save_steps=100, save_total_limit=1,
            seed=SEED, output_dir=f"{OUT}/checkpoints_{RUN_NAME}", report_to="none"))

    # loss only on the answer ("Long"/"Short"), not on the 3k-token prompt
    try:
        trainer = train_on_responses_only(trainer)
    except Exception:
        trainer = train_on_responses_only(trainer,
            instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
            response_part="<|start_header_id|>assistant<|end_header_id|>\n\n")
    lab = trainer.train_dataset[0]["labels"]
    target = tokenizer.decode([t for t in lab if t != -100])
    print("trained target of example 0:", repr(target))
    assert target.strip().startswith(("Long", "Short")), "response masking failed"
    assert st_train["truncated_share"] <= 0.02, "training prompts were cut - inspect the training log before continuing"

    # progress log (loss / ETA in the Kaggle log) + 12 h safety guard
    from transformers import TrainerCallback
    class Monitor(TrainerCallback):
        def __init__(self, budget_h):
            self.t0, self.budget, self.stopped_early = time.time(), budget_h * 3600, False
        def on_log(self, args, state, control, logs=None, **kw):
            if logs and "loss" in logs:
                el, done, tot = time.time() - self.t0, state.global_step, state.max_steps
                eta = el / done * (tot - done) / 3600 if done else 0.0
                print(f"step {done}/{tot}  loss {logs['loss']:.4f}  lr {logs.get('learning_rate', 0):.2e}  "
                      f"elapsed {el/3600:.2f} h  ETA {eta:.2f} h", flush=True)
        def on_step_end(self, args, state, control, **kw):
            if time.time() - self.t0 > self.budget and not self.stopped_early:
                self.stopped_early = True
                control.should_training_stop = True
                print(f"TIME BUDGET reached at step {state.global_step}/{state.max_steps}: stopping training, "
                      f"evaluation continues", flush=True)
    monitor = Monitor(TRAIN_BUDGET_HOURS)
    trainer.add_callback(monitor)

    t0 = time.time()
    stats = trainer.train()
    secs = time.time() - t0
    TRAIN_INFO = {"steps_done": int(stats.global_step), "steps_planned": int(trainer.state.max_steps),
                  "minutes": round(secs / 60, 1), "stopped_early_by_time_budget": monitor.stopped_early,
                  "final_train_loss": float(stats.training_loss)}
    print("training:", TRAIN_INFO)
    if MODE == "timing":
        sps = secs / max(stats.global_step, 1)
        print(f"\n~{sps:.1f} s/step at MAX_SEQ_LEN={MAX_SEQ_LEN}. Estimated full-epoch training time:")
        for n in [len(splits['train']), 4000, 3000, 2000]:
            print(f"  TRAIN_LIMIT={n:5d}: {L.estimate_hours(sps, n, BATCH, GRAD_ACC):5.1f} h (+ ~{(len(val_recs)+len(test_recs))*sps/GRAD_ACC/3/60:.0f} min evaluation)")
        print("Pick a TRAIN_LIMIT whose total stays under ~10 h, set MODE='train_eval', and commit again.")

## 5) Save adapter + evaluate (validation → threshold, test → final numbers)

In [ ]:
if MODE in ("pilot", "train_eval", "smoke"):
    if MODE == "train_eval":
        model.save_pretrained(f"{OUT}/adapter_{RUN_NAME}"); tokenizer.save_pretrained(f"{OUT}/adapter_{RUN_NAME}")
    FastLanguageModel.for_inference(model)
    y_val = [r["label"] for r in val_recs]
    p_val = SCORE(val_prompts)
    save_preds("val", val_recs, p_val)
    if MODE in ("pilot", "smoke"):              # test set NOT touched in the pilot / smoke test
        res = report(y_val, p_val, train_long_share=TRAIN_LONG_SHARE)
    else:
        y_test = [r["label"] for r in test_recs]
        p_test = SCORE(test_prompts)
        save_preds("test", test_recs, p_test)
        res = report(y_val, p_val, y_test, p_test, TRAIN_LONG_SHARE)
        res["majority_baseline"] = L.majority_baseline([r["label"] for r in splits["train"]], y_test)
    res["config"] = {k: globals()[k] for k in ["DATASET","MODE","SPLIT_MODE","MAX_SEQ_LEN","TRAIN_LIMIT","PILOT_N","BALANCE_TRAIN","VAL_LIMIT","SEED","LR","EPOCHS","BATCH","GRAD_ACC","LORA_R","LORA_ALPHA"]}
    res["n_train"] = len(train_recs)
    res["training"] = TRAIN_INFO
    res["data_file"] = DATA_FILE
    res["fp32_strategy"] = _STRATEGY
    json.dump(res, open(f"{OUT}/results_{RUN_NAME}.json", "w"), indent=2)
    show(res)
    if MODE == "train_eval":
        print(L.comparison_table({("Llama-3-8B-Instruct QLoRA (ours, argmax)", DATASET): res["test"]["argmax_0.5"],
                                  ("Llama-3-8B-Instruct QLoRA (ours, calibrated)", DATASET): res["test"]["calibrated"],
                                  ("Majority class (ours)", DATASET): res["majority_baseline"]}))
if MODE == "smoke":
    sps = TRAIN_INFO["minutes"] * 60 / max(TRAIN_INFO["steps_done"], 1)
    t_eval = len(val_recs) and (time.time() - t0 - TRAIN_INFO["minutes"] * 60) / len(val_recs)
    full_h = L.estimate_hours(sps, TRAIN_LIMIT, BATCH, GRAD_ACC) + t_eval * (VAL_LIMIT + len(test_recs)) / 3600
    print(f"\nSMOKE TEST PASSED. {sps:.0f} s/step, {t_eval:.1f} s per scored event.")
    print(f"Estimated MODE='train_eval' run: {full_h:.1f} h (must be under ~11 h on Kaggle)")
    print("distinct p_long values:", len(np.unique(np.round(p_val, 7))), "| exact 0.5 ties:", int(np.sum(p_val == 0.5)))
    assert full_h < 11.5, "full run would exceed Kaggle's 12 h limit - lower TRAIN_LIMIT and tell Claude"
